# Reddit Pipeline — CSCI 544

**How this notebook survives Colab dying:**
- All output lives in your Google Drive under `My Drive/CSCI544-Reddit/`
- The checkpoint (`checkpoint.json`) is written to Drive after every `.zst` file finishes
- If Colab dies, re-open this notebook, re-run all cells — it picks up where it left off

**Steps:**
1. `Cell 1` — Mount Drive + install libraries
2. `Cell 2` — Install torrent client
3. `Cell 3` — ⚠️ **ADD YOUR SUBREDDITS HERE** and download `.zst` files
4. `Cell 4` — All pipeline code (don't edit)
5. `Cell 5` — Run the pipeline

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Mount Google Drive + install libraries
# Run this first every time you open the notebook.
# You will get a Google sign-in popup — approve it.
# ════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q zstandard huggingface_hub

import os
from pathlib import Path

# All output (checkpoints, CSVs, zips) lives here in your Drive.
# It will be created automatically if it doesn't exist.
DRIVE_OUTPUT = Path('/content/drive/MyDrive/CSCI544-Reddit')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Colab working directory — .zst files are downloaded here (temporary)
ZST_DIR = Path('/content/reddit_zst')
ZST_DIR.mkdir(exist_ok=True)

print(f'✓ Drive mounted')
print(f'✓ Output folder: {DRIVE_OUTPUT}')
print(f'✓ Torrent download folder: {ZST_DIR}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Install torrent client
# Only needs to run once per Colab session.
# ════════════════════════════════════════════════════════════

!apt-get install -q -y transmission-cli transmission-daemon
print('✓ Transmission torrent client installed')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — ⚠️ ADD YOUR SUBREDDITS HERE
#
# For each subreddit you want, add a line to TORRENT_FILES.
# Format:
#   ("filename_inside_torrent.zst", "subreddit_name", "submission" or "comment")
#
# The filename is the exact name of the file inside the torrent.
# Check the torrent file list in qBittorrent to get exact names.
# They usually follow the pattern:  subredditname_submissions.zst
#                                   subredditname_comments.zst
#
# The magnet link below downloads the ENTIRE torrent index first,
# then selects only the files you list in TORRENT_FILES.
# ════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────
# ⚠️  EDIT THIS LIST — one tuple per file you want
# ──────────────────────────────────────────────────────────
TORRENT_FILES = [
    # ("exact_filename_in_torrent.zst",  "subreddit_name",  "submission" or "comment")

    ("technology_submissions.zst",  "technology",  "submission"),
    ("technology_comments.zst",     "technology",  "comment"),

    # Add more rows below — uncomment or copy the pattern:
    # ("science_submissions.zst",     "science",     "submission"),
    # ("science_comments.zst",        "science",     "comment"),
    # ("news_submissions.zst",        "news",        "submission"),
    # ("news_comments.zst",           "news",        "comment"),
    # ("worldnews_submissions.zst",   "worldnews",   "submission"),
    # ("investing_submissions.zst",   "investing",   "submission"),
    # ("movies_submissions.zst",      "movies",      "submission"),
    # ("books_submissions.zst",       "books",       "submission"),
]
# ──────────────────────────────────────────────────────────

MAGNET = (
    "magnet:?xt=urn:btih:3E3F64DEE22DC304CDD2546254CA1F8E8AE542B4&dn=reddit"
    "&tr=https%3A%2F%2Facademictorrents.com%2Fannounce.php%3Fpasskey%3D1489287c03868c5a5e6d87af166c32ca"
    "&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337%2Fannounce"
)

# Download each file using transmission-cli
# Each file is downloaded individually so we only grab what we need.
import subprocess

for filename, subreddit, post_type in TORRENT_FILES:
    dest = ZST_DIR / filename
    if dest.exists():
        print(f'✓ Already downloaded: {filename}')
        continue

    print(f'\n⏳ Downloading {filename} ...')
    # transmission-cli will fetch metadata then select the matching file
    result = subprocess.run([
        'transmission-cli',
        MAGNET,
        '--download-dir', str(ZST_DIR),
        '--find', filename,     # only download this file from the torrent
        '--finish-call', 'echo done',
    ], capture_output=False, text=True, timeout=7200)   # 2hr timeout per file

    if dest.exists():
        size_gb = dest.stat().st_size / 1e9
        print(f'✓ Downloaded: {filename} ({size_gb:.1f} GB)')
    else:
        print(f'⚠ Download may not have completed: {filename}')
        print('  Check the output above. You can re-run this cell to retry.')

print('\n── Download summary ──')
for filename, _, _ in TORRENT_FILES:
    dest = ZST_DIR / filename
    status = f'{dest.stat().st_size/1e9:.1f} GB' if dest.exists() else 'MISSING'
    print(f'  {filename:<45s}  {status}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Pipeline code
# Don't edit this cell. Just run it to load all the functions.
# ════════════════════════════════════════════════════════════

import zstandard as zstd
import json
import io
import zipfile
import random
import shutil
from datetime import datetime, timezone
from collections import defaultdict

import pandas as pd

try:
    from huggingface_hub import HfApi
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False

# ── Settings ─────────────────────────────────────────────────
MIN_WORDS   = 50
MAX_WORDS   = 1_500
RANDOM_SEED = 42

# Hugging Face (optional) — paste your token here or leave blank
HF_REPO_ID = "your-hf-username/reddit-ai-detection"
HF_TOKEN   = ""   # ← paste hf_... token here if you want HF upload

# These paths come from Cell 1 — don't change
OUTPUT_DIR      = DRIVE_OUTPUT           # everything written directly to Drive
CHECKPOINT_FILE = OUTPUT_DIR / 'checkpoint.json'

# ── Domain map ───────────────────────────────────────────────
# Maps subreddit name → domain label used in the dataset.
# If you add a subreddit in Cell 3 that isn't listed here,
# add it below so it gets a proper domain label.
SUBREDDIT_DOMAIN_MAP = {
    # Technology
    'technology':        'technology',
    'programming':       'technology',
    'MachineLearning':   'technology',
    'artificial':        'technology',
    'compsci':           'technology',
    'hardware':          'technology',
    'cybersecurity':     'technology',
    # News
    'news':              'news',
    'worldnews':         'news',
    'politics':          'news',
    'neutralnews':       'news',
    'UpliftingNews':     'news',
    # Science
    'science':           'science',
    'askscience':        'science',
    'EverythingScience': 'science',
    'biology':           'science',
    'physics':           'science',
    # Finance
    'investing':         'finance',
    'economics':         'finance',
    'wallstreetbets':    'finance',
    'personalfinance':   'finance',
    'stocks':            'finance',
    # Entertainment
    'movies':            'entertainment',
    'books':             'entertainment',
    'television':        'entertainment',
    'music':             'entertainment',
    'gaming':            'entertainment',
}

# ── Year buckets ─────────────────────────────────────────────
PRE_BUCKETS = {
    (2005, 2010):  1_000,
    (2011, 2015):  3_000,
    (2016, 2018):  5_000,
    (2019, 2021): 11_000,
}
POST_BUCKETS = {
    (2022, 2022):  3_000,
    (2023, 2023):  6_500,
    (2024, 2024):  6_500,
    (2025, 2026):  4_000,
}
ALL_BUCKETS = {**PRE_BUCKETS, **POST_BUCKETS}


# ── Utilities ────────────────────────────────────────────────
def get_bucket(year):
    for (start, end) in ALL_BUCKETS:
        if start <= year <= end:
            return (start, end)
    return None

def word_count(text):
    return len(text.split())

def length_bin(wc):
    if wc < 100:  return 'short'
    if wc < 300:  return 'medium'
    if wc < 700:  return 'long'
    return 'very_long'

def extract_text(obj, post_type):
    if post_type == 'submission':
        return obj.get('selftext', '').strip()
    return obj.get('body', '').strip()

def passes_filters(text):
    if not text or text in {'[deleted]', '[removed]', ''}:
        return False
    wc = word_count(text)
    return MIN_WORDS <= wc <= MAX_WORDS

def read_lines_zst(file_path):
    with open(file_path, 'rb') as f:
        dctx = zstd.ZstdDecompressor(max_window_size=2**31)
        reader = dctx.stream_reader(f)
        text_stream = io.TextIOWrapper(reader, encoding='utf-8')
        for line in text_stream:
            yield line


# ── Checkpoint ───────────────────────────────────────────────
def save_checkpoint(completed_files, reservoir):
    """Save progress to Drive. Called after every .zst file."""
    serialisable = {str(k): v for k, v in reservoir.items()}
    payload = {'completed_files': completed_files, 'reservoir': serialisable}
    tmp = OUTPUT_DIR / 'checkpoint.tmp.json'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False)
    # Atomic replace so a crash mid-write doesn't corrupt the checkpoint
    tmp.replace(CHECKPOINT_FILE)
    size_kb = CHECKPOINT_FILE.stat().st_size // 1024
    print(f'  💾 Checkpoint saved to Drive ({size_kb:,} KB) → {CHECKPOINT_FILE}')

def load_checkpoint():
    """Load checkpoint from Drive if it exists."""
    if not CHECKPOINT_FILE.exists():
        print('  No checkpoint found — starting fresh.')
        return [], {}
    with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
        payload = json.load(f)
    completed_files = payload.get('completed_files', [])
    reservoir = {}
    for k_str, records in payload.get('reservoir', {}).items():
        try:
            reservoir[eval(k_str)] = records
        except Exception:
            continue
    total = sum(len(v) for v in reservoir.values())
    print(f'  📂 Checkpoint loaded from Drive — '
          f'{len(completed_files)} file(s) done, {total:,} records in reservoir.')
    return completed_files, reservoir


# ── Reservoir sampler ─────────────────────────────────────────
class BucketCollector:
    def __init__(self, quotas, domains, preloaded_reservoir=None):
        self.domains = domains
        n_domains = max(1, len(domains))
        self.cell_target = {}
        for bucket, total in quotas.items():
            per_domain = max(1, total // n_domains)
            for d in domains:
                self.cell_target[(d, bucket)] = per_domain
        if preloaded_reservoir:
            self.reservoir  = defaultdict(list, preloaded_reservoir)
            self.seen_count = defaultdict(int, {k: len(v) for k, v in self.reservoir.items()})
        else:
            self.reservoir  = defaultdict(list)
            self.seen_count = defaultdict(int)

    def try_add(self, record):
        key    = (record['domain'], record['bucket'])
        target = self.cell_target.get(key, 0)
        if target == 0:
            return
        n = self.seen_count[key]
        self.seen_count[key] += 1
        reservoir = self.reservoir[key]
        if len(reservoir) < target:
            reservoir.append(record)
        else:
            r = random.randint(0, n)
            if r < target:
                reservoir[r] = record

    def total_collected(self):
        return sum(len(v) for v in self.reservoir.values())

    def status_table(self):
        rows = []
        for (domain, bucket), target in sorted(self.cell_target.items()):
            have = len(self.reservoir[(domain, bucket)])
            bar  = '█' * int(20 * have / max(target, 1))
            rows.append(f'  {domain:<15s} {str(bucket):<20s} {have:>5d}/{target:<5d}  {bar}')
        return '\n'.join(rows)

    def to_dataframe(self):
        all_records = [r for records in self.reservoir.values() for r in records]
        df = pd.DataFrame(all_records)
        if 'bucket' in df.columns:
            df = df.drop(columns=['bucket'])
        return df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)


# ── Hugging Face upload ───────────────────────────────────────
def upload_to_hf(local_path, repo_path):
    if not HF_AVAILABLE or not HF_TOKEN or 'your-hf-username' in HF_REPO_ID:
        print('  HF upload skipped (token or repo not configured).')
        return
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type='dataset', token=HF_TOKEN, exist_ok=True)
    api.upload_file(path_or_fileobj=str(local_path), path_in_repo=repo_path,
                    repo_id=HF_REPO_ID, repo_type='dataset', token=HF_TOKEN)
    print(f'  ✓ Uploaded to HF: hf://datasets/{HF_REPO_ID}/{repo_path}')


print('✓ All pipeline functions loaded.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Run the pipeline
# This is the long-running cell. Leave it running.
# If Colab dies, re-run Cells 1-4 then re-run this cell.
# It will resume from the last completed .zst file.
# ════════════════════════════════════════════════════════════

random.seed(RANDOM_SEED)

# Build INPUT_FILES from TORRENT_FILES defined in Cell 3
INPUT_FILES = [
    (str(ZST_DIR / filename), subreddit, post_type)
    for filename, subreddit, post_type in TORRENT_FILES
]

# ── Startup: load checkpoint from Drive ──────────────────────
print('═' * 65)
print(' STARTUP — Checking Drive for existing checkpoint')
print('═' * 65)
completed_files, preloaded_reservoir = load_checkpoint()

domains   = sorted(set(SUBREDDIT_DOMAIN_MAP.values()))
collector = BucketCollector(ALL_BUCKETS, domains, preloaded_reservoir or None)

# ── Phase 1: Read & collect ───────────────────────────────────
print()
print('═' * 65)
print(' PHASE 1 — Reading .zst files')
print('═' * 65)

for file_path, subreddit, post_type in INPUT_FILES:

    if file_path in completed_files:
        print(f'\n✓  Already done (checkpoint): {file_path}')
        continue

    if not Path(file_path).exists():
        print(f'\n⚠  File not found — did the torrent finish? Skipping: {file_path}')
        continue

    domain = SUBREDDIT_DOMAIN_MAP.get(subreddit, 'other')
    print(f'\n▶  {file_path}')
    print(f'   subreddit={subreddit}  domain={domain}  type={post_type}')

    lines_read = 0
    for line in read_lines_zst(file_path):
        lines_read += 1
        try:
            obj  = json.loads(line)
            text = extract_text(obj, post_type)
            if not passes_filters(text):
                continue
            ts      = obj.get('created_utc', 0)
            created = datetime.fromtimestamp(ts, tz=timezone.utc)
            year    = created.year
            bucket  = get_bucket(year)
            if bucket is None:
                continue
            wc = word_count(text)
            record = {
                'text':        text,
                'subreddit':   subreddit,
                'domain':      domain,
                'post_type':   post_type,
                'year':        year,
                'bucket':      bucket,
                'word_count':  wc,
                'length_bin':  length_bin(wc),
                'score':       obj.get('score', 0),
                'created_utc': int(ts),
                'id':          obj.get('id', ''),
            }
            collector.try_add(record)
        except Exception:
            continue

        if lines_read % 250_000 == 0:
            print(f'   Lines: {lines_read:>10,} | Collected: {collector.total_collected():>7,}')

    print(f'   Done. Lines read: {lines_read:,} | Collected so far: {collector.total_collected():,}')

    # ✅ Save checkpoint to Drive immediately after each file
    completed_files.append(file_path)
    save_checkpoint(completed_files, dict(collector.reservoir))

print('\n── Collection status ──────────────────────────────────')
print(collector.status_table())

# ── Phase 2: Save datasets to Drive ──────────────────────────
print()
print('═' * 65)
print(' PHASE 2 — Saving datasets to Drive')
print('═' * 65)

df      = collector.to_dataframe()
pre_df  = df[df['year'] <  2022].copy().reset_index(drop=True)
post_df = df[df['year'] >= 2022].copy().reset_index(drop=True)

print(f'  Pre-2022  : {len(pre_df):,} records')
print(f'  Post-2022 : {len(post_df):,} records')
print(f'  Total     : {len(df):,} records')

zip_files = []
for split_name, split_df in [('pre_2022', pre_df), ('post_2022', post_df), ('combined', df)]:
    csv_path = OUTPUT_DIR / f'reddit_{split_name}.csv'
    zip_path = OUTPUT_DIR / f'reddit_{split_name}.zip'
    split_df.to_csv(csv_path, index=False)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
        zf.write(csv_path, arcname=csv_path.name)
    size_kb = zip_path.stat().st_size // 1024
    print(f'  ✓ Saved to Drive: {zip_path.name}  ({size_kb:,} KB)')
    zip_files.append(zip_path)

# ── Phase 3: Hugging Face ─────────────────────────────────────
print()
print('═' * 65)
print(' PHASE 3 — Hugging Face upload (skipped if not configured)')
print('═' * 65)
for zip_path in zip_files:
    upload_to_hf(zip_path, f'data/{zip_path.name}')

# ── Summary ───────────────────────────────────────────────────
print()
print('═' * 65)
print(' SUMMARY')
print('═' * 65)
print(f'  Total     : {len(df):,}')
print(f'  Pre-2022  : {len(pre_df):,}')
print(f'  Post-2022 : {len(post_df):,}\n')
print('  By domain:')
for domain, grp in df.groupby('domain'):
    print(f'    {domain:<18s} {len(grp):>7,}')
print('\n  By year:')
for year, count in df.groupby('year').size().items():
    bar = '▪' * min(40, count // 100)
    print(f'    {year}  {count:>6,}  {bar}')
print('\n  By length bin:')
for bin_name, grp in df.groupby('length_bin'):
    print(f'    {bin_name:<12s} {len(grp):>7,}')
print(f'\n✓ Pipeline complete. Files saved to: {OUTPUT_DIR}')